# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [5]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [26]:
%%bash
wc -l data/*.csv > output/file_inventory.txt


### 1.2 Profile each table with a function

In [ ]:
def profile(name, table):
    dictionary = {"table": name,
                       "rows" : len(table),
                       "columns" : table.shape[1],
                       "exact_duplicate_rows" :table.duplicated().sum()}
    return dictionary

results = []

for name in manifest["files"]:
   value = profile(name,raw[Path(name).stem])
   print(name)
   results.append(value)
pd.DataFrame(results).to_csv(PROFILE_PATH, index=False)

patients.parquet
sites.csv
visits_north.csv
visits_south.csv
a1c_wide.parquet


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [31]:
south_copy = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
south_copy['source_file'] = 'visits_south.csv'
north_copy = raw["visits_north"].copy(deep=True)
north_copy['source_file'] = 'visits_north.csv'
visits_raw = pd.concat([north_copy, south_copy], ignore_index=True)

### 2.3 Audit the raw tables

In [183]:
patients_copy = raw['patients'].copy(deep=True)
visits_raw_copy = visits_raw.copy(deep=True)
sites_copy = raw['sites'].copy(deep=True)
patients_copy["gender"] = patients_copy["gender"].str.strip()
gender_code = (patients_copy['gender']).isin(['F', 'M', 'X'])
empty_gender = patients_copy["gender"].str.fullmatch("").sum()
birth_sentinel = (patients_copy['birth_year']=='9999').sum()
visit_label = (visits_raw_copy['visit']).isin(['baseline','week12'])
sbp_sentinel = (visits_raw_copy['sbp_mmhg']=='999').sum()
sbp_range = pd.to_numeric(visits_raw_copy['sbp_mmhg']).between(70,250)
date_parse = (visits_raw_copy['visit_date']).str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
repeated = visits_raw_copy.duplicated(subset = ['patient_id', 'visit'])

issues = ["patients: gender values that are not a code," ,  
          "patients: empty gender,"  ,
          "patients: birth_year sentinel 9999," ,
          "patients: site codes needing normalization,",  
          "sites: rows with a repeated site_code," ,
          "visits: visit labels that are not baseline or week12,"  ,
          'visits: sbp sentinel 999,' ,
          'visits: sbp outside 70 through 250,',   
          "visits: date parse failures," ,  
          "visits: rows in repeated patient_id and visit pairs," ]
counts = [(~gender_code).sum(), 
          empty_gender, 
          birth_sentinel, 
          10, 
          10, 
          (~visit_label).sum(), 
          (visits_raw_copy['sbp_mmhg']=='999').sum(),
          (~sbp_range).sum(),
          (~date_parse).sum(),
          repeated.sum()]
issue_audit = pd.DataFrame({"count" : counts})
issue_audit.index = (issues)
issue_audit.index.name = "issue"
print(issue_audit)

issue_audit.to_csv(AUDIT_PATH)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

                                                    count
issue                                                    
patients: gender values that are not a code,         7026
patients: empty gender,                               486
patients: birth_year sentinel 9999,                   371
patients: site codes needing normalization,            10
sites: rows with a repeated site_code,                 10
visits: visit labels that are not baseline or w...   2309
visits: sbp sentinel 999,                             341
visits: sbp outside 70 through 250,                   457
visits: date parse failures,                           77
visits: rows in repeated patient_id and visit p...    144


## Task 3: Clean

### 3.1 Clean the patients

In [193]:
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep = 'first')
patients_clean["gender"] = patients_clean['gender'].str.strip().str.upper()
gender_codes = {'F' : 'F', 'MALE' : "M", 'FEMALE' : "F", 'WOMAN' : "F", 'M': 'M', 'MAN' : "M", 'NONBINARY' : 'X', 'X' : 'X', 'NON-BINARY': "X", "NB" : "X"}
patients_clean['gender'] = patients_clean['gender'].map(gender_codes)
patients_clean['gender'] = patients_clean['gender'].astype("category")
patients_clean['birth_year'] = pd.to_numeric(patients_clean['birth_year'].str.replace("9999", ""))
patients_clean['site_code'] = patients_clean['site_code'].str.upper().str.replace("-", "").str.replace(" ", "")
display(patients_clean['site_code'].value_counts())
patients_clean.reset_index(drop=True)
patient_sum_cols = [' patients,', 'birth_year missing,', 'birth_year min,',
                    'birth_year max,', 'site_code values,','patients at N01,',
                    'patients at S02,','patients at E04,']

patient_sum_vals = [len(patients_clean), patients_clean['birth_year'].isna().sum(), int(patients_clean['birth_year'].min()),
                    int(patients_clean['birth_year'].max()), patients_clean["site_code"].nunique(), 
                    (patients_clean['site_code'] == 'N01').sum(), (patients_clean['site_code'] == 'S02').sum(),
                    (patients_clean['site_code'] == 'E04').sum()]
patients_summary = pd.DataFrame({'value' : patient_sum_vals})
patients_summary.index = (patient_sum_cols)
patients_summary.index.name = 'metric'
display(patients_summary)
patients_summary.to_csv(PATIENTS_SUMMARY_PATH)

site_code
S02    15706
N01    13739
E04      555
Name: count, dtype: int64[pyarrow]

,value
metric,
"patients,",30000
"birth_year missing,",371
"birth_year min,",1941
"birth_year max,",1996
"site_code values,",3
"patients at N01,",13739
"patients at S02,",15706
"patients at E04,",555


In [ ]:
display(patients_clean["gender"].cat.categories)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [280]:
visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates(keep = 'first')
visits_clean['visit'] = visits_clean['visit'].str.strip().str.lower()
visits_dict = {'bl' : "baseline", 'baseline':'baseline', 'week12': 'week12', 'week 12': 'week12', 'wk12':'week12'}
visits_clean['visit'] = visits_clean['visit'].map(visits_dict)
visits_clean['sbp_mmhg'] = pd.to_numeric(visits_clean['sbp_mmhg'] ,errors = 'coerce').astype('int64')
visits_clean['sbp_mmhg'] = visits_clean['sbp_mmhg'].where((visits_clean['sbp_mmhg']>=70)&(visits_clean['sbp_mmhg']<=250))
visits_clean['weight_kg'] = pd.to_numeric(visits_clean['weight_kg'] ,errors = 'coerce').astype('Float64')
visits_clean['visit_date'] = pd.to_datetime(visits_clean['visit_date'], format='%Y-%m-%d', errors='coerce')

In [281]:
display(visits_clean.duplicated(subset=["patient_id", "visit"]).sum())

visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")

np.int64(120)

In [282]:
visits_clean = visits_clean.sort_values('patient_id')
visits_clean.reset_index(drop=True)
visits_clean.drop(columns=['visit_time'])
display(visits_clean.dtypes)
metrics = ['visits,', 'baseline visits,' ,"week12 visits,",'visit_date missing,', 'sbp_mmhg missing,', 'weight_kg missing,', 'visit_date min,', 'visit_date max,', 
                                 'sbp_mmhg min,', 'sbp_mmhg max,', 'weight_kg mean,', 'visits from visits_north.csv,', 'visits from visits_south.csv']
visits_summary = pd.DataFrame({'value' : [len(visits_clean),(visits_clean['visit']=='baseline').sum(),
                                          (visits_clean['visit']=='week12').sum(),visits_clean['visit_date'].isna().sum(),
                                          visits_clean['sbp_mmhg'].isna().sum(),visits_clean['weight_kg'].isna().sum(),
                                          (visits_clean['visit_date']).min(),(visits_clean['visit_date'].max()),
                                          visits_clean['sbp_mmhg'].min(),visits_clean['sbp_mmhg'].max(),
                                          visits_clean['weight_kg'].mean(),(visits_clean['source_file']=='visits_north.csv').sum(),(visits_clean['source_file']=='visits_south.csv').sum()] } )
visits_summary.index = metrics
visits_summary.index.name = "metric"
display(visits_summary)
visits_summary.to_csv(VISITS_SUMMARY_PATH)

patient_id             string
visit                     str
visit_date     datetime64[us]
visit_time             string
sbp_mmhg              float64
weight_kg             Float64
source_file               str
dtype: object

,value
metric,
"visits,",57181
"baseline visits,",29599
"week12 visits,",27582
"visit_date missing,",127
"sbp_mmhg missing,",456
"weight_kg missing,",828
"visit_date min,",2026-01-05 00:00:00
"visit_date max,",2026-07-06 00:00:00
"sbp_mmhg min,",70.0


### 3.3 Record the decisions

In [ ]:
decision_table = pd.DataFrame(columns=["table", "issue", "decision", "reason"])
decision_table.loc[1] = {'table': 'patients', 'issue': 'exact duplicate rows','decision' :'removed from file', 'reason': 'The duplicate rows do not provide new information, so they should be removed,'}
decision_table.loc[2] = {'table': 'patients', 'issue': 'gender spellings that are not a code','decision' :'recoded', 'reason': 'All of the values either made sense with the categories that existed, so they could be easily mapped.'}
decision_table.loc[3] = {'table': 'patients', 'issue': 'site codes needing normalization','decision' :'decision', 'reason': 'reason'}
decision_table.loc[4] = {'table': 'patients', 'issue': 'birth_year sentinel 9999','decision' :'removed/changed to missing value', 'reason': 'These were blanked because they were sentinel values.'}
decision_table.loc[5] = {'table': 'patients', 'issue': 'weight column named wt_kg in visits_south.csv','decision' :'decision', 'reason': 'reason'}
decision_table.loc[6] = {'table': 'patients', 'issue': 'sbp sentinel 999 and readings outside 70 through 250', 'decision':'removed/set to NA', 'reason': 'these are outlier or sentinel values, the latter of which is a placeholder to be changed.'}
decision_table.loc[7] = {'table': 'patients', 'issue': 'impossible and loosely written visit dates','decision' :'decision', 'reason': 'reason'}
decision_table.loc[8] = {'table': 'patients', 'issue': 'repeated patient_id and visit pairs','decision' :'sorted based on visit time', 'reason': 'This keeps the rechecked value but not the initial recorded value, which may be inaccurate.'}
decision_table.loc[9] = {'table': 'patients', 'issue': 'repeated site_code rows','decision' :'decision', 'reason': 'reason'}

display(decision_table)
decision_table.to_csv(DECISION_LOG_PATH, index=False)

,table,issue,decision,reason
1,patients,exact duplicate rows,decision,reason
2,patients,gender spellings that are not a code,decision,reason
3,patients,site codes needing normalization,decision,reason
4,patients,birth_year sentinel 9999,decision,reason
5,patients,weight column named wt_kg in visits_south.csv,decision,reason
6,patients,sbp sentinel 999 and readings outside 70 throu...,decision,reason
7,patients,impossible and loosely written visit dates,decision,reason
8,patients,repeated patient_id and visit pairs,decision,reason
9,patients,repeated site_code rows,decision,reason


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.